# A guide Portfolio Optimization Environment

This notebook aims to provide an example of using PortfolioOptimizationEnv (or POE) to train a reinforcement learning model that learns to solve the portfolio optimization problem.

In this document, we will reproduce a famous architecture called EIIE (ensemble of identical independent evaluators), introduced in the following paper:

- Zhengyao Jiang, Dixing Xu, & Jinjun Liang. (2017). A Deep Reinforcement Learning Framework for the Financial Portfolio Management Problem. https://doi.org/10.48550/arXiv.1706.10059.

It's advisable to read it to understand the algorithm implemented in this notebook.

### Note
If you're using this environment, consider citing the following paper (in adittion to FinRL references):

- Caio Costa, & Anna Costa (2023). POE: A General Portfolio Optimization Environment for FinRL. In *Anais do II Brazilian Workshop on Artificial Intelligence in Finance* (pp. 132–143). SBC. https://doi.org/10.5753/bwaif.2023.231144.

```
@inproceedings{bwaif,
 author = {Caio Costa and Anna Costa},
 title = {POE: A General Portfolio Optimization Environment for FinRL},
 booktitle = {Anais do II Brazilian Workshop on Artificial Intelligence in Finance},
 location = {João Pessoa/PB},
 year = {2023},
 keywords = {},
 issn = {0000-0000},
 pages = {132--143},
 publisher = {SBC},
 address = {Porto Alegre, RS, Brasil},
 doi = {10.5753/bwaif.2023.231144},
 url = {https://sol.sbc.org.br/index.php/bwaif/article/view/24959}
}

```

## Installation and imports

To run this notebook in google colab, uncomment the cells below.

In [1]:
## install finrl library
# !sudo apt install swig
# !pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

In [2]:
## We also need to install quantstats, because the environment uses it to plot graphs
# !pip install quantstats

In [2]:
## Hide matplotlib warnings
# import warnings
# warnings.filterwarnings('ignore')

import logging
logging.getLogger('matplotlib.font_manager').disabled = True

#### Import the necessary code libraries

In [3]:
import pandas as pd
import yfinance as yf
from datetime import datetime
class YahooRealtimeDownloader:
    """
    Provides methods for retrieving near-real-time (1-minute) stock data
    from Yahoo Finance API, returning only the last available bar or
    "one minute behind" the latest bar.
    """

    def __init__(self, ticker_list: list):
        """
        Parameters
        ----------
        ticker_list: list
            a list of stock tickers
        """
        self.ticker_list = ticker_list

    def fetch_data(self, proxy=None, pick_second_to_last=True) -> pd.DataFrame:
        """
        Fetches near-real-time 1-minute data from Yahoo API for the current day.
        """
        import datetime
        data_df = pd.DataFrame()
        num_failures = 0

        # 按照 1 分钟周期下载当日数据
        for tic in self.ticker_list:
            temp_df = yf.download(
                tickers=tic,
                period='1d',
                interval='1m',
                proxy=proxy,
                progress=False
            )

            temp_df["tic"] = tic

            if len(temp_df) > 0:
                # 如果您想获取倒数第二条数据
                if pick_second_to_last and len(temp_df) > 1:
                    temp_df = temp_df.iloc[[-2]]  # 取倒数第二行
                else:
                    temp_df = temp_df.iloc[[-1]]

                data_df = pd.concat([data_df, temp_df], axis=0)
            else:
                num_failures += 1

        if num_failures == len(self.ticker_list):
            raise ValueError("No data is fetched. Possibly all tickers returned empty for today.")

        # reset the index
        data_df = data_df.reset_index()

        # rename columns
        try:
            data_df.columns = [
                "date",
                "open",
                "high",
                "low",
                "close",
                "adjcp",
                "volume",
                "tic",
            ]
            data_df["close"] = data_df["adjcp"]
            data_df = data_df.drop(labels="adjcp", axis=1)
        except ValueError:
            print("Columns might not match the expected format; please check yfinance returned columns.")

        # 将日期列转换为 datetime
        data_df["date"] = pd.to_datetime(data_df["date"])
        
        # 获取本地当前日期，例如 2023-10-10
        today = datetime.date.today()

        # 如果行里的 date 不是当天，就改为当天；去掉时分秒，保留 YYYY-MM-DD
        def fix_date(dt):
            if dt.date() != today:
                return today.strftime('%Y-%m-%d')
            else:
                return dt.strftime('%Y-%m-%d')

        data_df["date"] = data_df["date"].apply(fix_date)

        # 再创建 day 列，这里仅保留日期而无时分秒，所以先转回 datetime
        data_df["date"] = pd.to_datetime(data_df["date"])
        data_df["day"] = data_df["date"].dt.dayofweek
        data_df["date"] = data_df["date"].dt.strftime("%Y-%m-%d")

        data_df = data_df.dropna().reset_index(drop=True)

        data_df = data_df.sort_values(by=["date", "tic"]).reset_index(drop=True)

        print("Shape of realtime DataFrame: ", data_df.shape)
        print(data_df)
        return data_df

In [4]:
import torch

from sklearn.preprocessing import MaxAbsScaler

import sys
import os

# 获取当前文件的绝对路径，并向上追溯两级到项目根目录
project_root = os.path.dirname(os.path.abspath(os.getcwd()))
print(project_root)
sys.path.append(project_root)

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import GroupByScaler
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv
from finrl.agents.portfolio_optimization.models import DRLAgent
from finrl.agents.portfolio_optimization.architectures import EIIE

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

/home/stock/projects/finstock/FinRL


/mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/torch_geometric/typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/torch_sparse/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "
/mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.

## Fetch data

In his paper, *Jiang et al* creates a portfolio composed by the top-11 cryptocurrencies based on 30-days volume. Since it's not specified when this classification was done, it's difficult to reproduce, so we will use a similar approach in the Brazillian stock market:

- We select top-10 stocks from Brazillian stock market;
- For simplicity, we disconsider stocks that have missing data for the days in period 2011-01-01 to 2019-12-31 (9 years);

In [5]:
market = "us"
if market.lower() == "us":
    TOP_BRL = [
        'BILI','NIO', 'JD', 'YINN', 'YANG', 'FUTU','BABA','PDD'
    ]
elif market.lower() == "hk":
    TOP_BRL = [
         '1812.hk', '3900.hk', '2777.hk', '1810.hk', '2878.HK','0029.hk'
    ]
elif market.lower() == "jp":
    TOP_BRL = [
        '8031.T', '8058.T','8001.T','8053.T','8002.T'
    ]
elif market.lower() == "ch":
    TOP_BRL = [
         '603063.ss', '603319.ss', '000657.sz','002640.sz','002664.sz','300182.sz','688200.SS','002850.SZ'
    ]

    # TOP_BRL = [
    # '000063.SZ',
    # '002068.SZ',
    # '002138.SZ',
    # '002779.SZ',
    # '002850.SZ',
    # '300408.SZ',
    # '300442.SZ',
    # '300657.SZ',
    # '300673.SZ',
    # '300909.SZ',
    # '300913.SZ',
    # '601111.SS',
    # '601689.SS',
    # '603063.SS',
    # '603236.SS',
    # '603305.SS',
    # '603556.SS',
    # '603667.SS',
    # '688088.SS',
    # '688160.SS',
    # '688200.SS']
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

print("当前选择的市场:", market)
print("股票列表:", TOP_BRL)
# '9988.hk',,'1797.hk''1918.hk', '3319.hk',

当前选择的市场: us
股票列表: ['BILI', 'NIO', 'JD', 'YINN', 'YANG', 'FUTU', 'BABA', 'PDD']


In [8]:
print(len(TOP_BRL))

portfolio_raw_df = YahooDownloader(start_date = '2021-01-01',
                                end_date = '2025-05-28',
                                ticker_list = TOP_BRL).fetch_data()
portfolio_raw_df = portfolio_raw_df.drop('day', axis=1)

8


[*********************100%***********************]  1 of 1 completed

1 Failed download:
['BILI']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['NIO']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['JD']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['YINN']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['YANG']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['FUTU']: YFRateLimitError('Too Many Requests. Rate limited

ValueError: no data is fetched.

In [6]:
# realtime_df = YahooRealtimeDownloader(ticker_list = TOP_BRL).fetch_data()
# # 假设这两个 DataFrame 的列名相同 
# portfolio_raw_df = pd.concat([portfolio_raw_df, realtime_df], ignore_index=True)

# 对合并后的数据，根据日期和股票标的排序，并重新索引
portfolio_raw_df = portfolio_raw_df.sort_values(["date", "tic"]).reset_index(drop=True)

# 检查合并后的数据
print(portfolio_raw_df.head())
print(portfolio_raw_df.tail())


NameError: name 'portfolio_raw_df' is not defined

In [7]:
from finrl.config import INDICATORS
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
fe = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=INDICATORS,
    use_vix=True,
    use_turbulence=True,
    user_defined_feature=False
)
processed = fe.preprocess_data(portfolio_raw_df)
logging.info("数据预处理完成。")
processed.tail()

NameError: name 'portfolio_raw_df' is not defined

In [11]:
from finrl.config import INDICATORS
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
fe_without_vix = FeatureEngineer(
    use_technical_indicator=True,
    tech_indicator_list=INDICATORS,
    use_vix=False,
    use_turbulence=True,
    user_defined_feature=False
)
processed_without_vix = fe_without_vix.preprocess_data(portfolio_raw_df)
logging.info("数据预处理完成。")
processed_without_vix.tail()

Successfully added technical indicators
Successfully added turbulence index


,date,close,high,low,open,volume,tic,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,turbulence
8699,2025-05-02,34.025002,34.410000,33.860001,34.380001,4565177,JD,-1.553459,37.706237,30.974351,44.580447,-54.459306,10.153180,36.361803,38.365017,2.351354
8700,2025-05-02,4.045000,4.150000,4.025000,4.080000,10988919,NIO,0.015043,4.413317,3.027183,49.325429,54.177562,2.178078,3.828167,4.181917,2.351354
8701,2025-05-02,109.820000,110.489998,109.084999,109.510002,3189956,PDD,-1.473252,110.853742,86.656258,50.675194,20.475700,4.504246,106.522333,113.533167,2.351354
8702,2025-05-02,38.889999,38.980000,37.890099,37.980000,1164602,YANG,-0.925551,61.042564,32.698436,43.924083,-73.529319,4.631570,44.208267,42.956502,2.351354
8703,2025-05-02,34.639999,35.380001,34.535099,35.290001,2807505,YINN,-1.259180,36.575976,23.489024,49.494761,7.377176,3.843887,34.159365,38.405668,2.351354


In [12]:
# from feature.ch_feature_engineer import ChFeatureEngineer
# ch_fe = ChFeatureEngineer()
# portfolio_processed = ch_fe.preprocess_data(processed)
# logging.info("自定义特征工程完成。")
# 横向合并数据集（保留完整时间序列）
merged_df = pd.concat(
    [processed_without_vix, processed[['vix']]],  # 只取vix列
    axis=1
)

# 处理可能的缺失值（优先用原始vix数据，缺失处用前后填充）
merged_df['vix'] = merged_df['vix'].fillna(method='ffill').fillna(method='bfill')

# 验证合并结果
logging.info(f"合并后数据集形状：{merged_df.shape}")
merged_df.tail()
processed=merged_df

/tmp/ipykernel_178947/3172080016.py:12: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged_df['vix'] = merged_df['vix'].fillna(method='ffill').fillna(method='bfill')


In [25]:
import datetime
import pandas as pd

def get_holiday_date_ranges(holiday_dict, days_before=30, days_after=10):
    """
    生成节假日前后指定天数的日期范围
    
    参数:
        holiday_dict (dict): 节假日字典，格式为 {'节日名称': {'年份': '日期'}}
        days_before (int): 节前天数，默认30天
        days_after (int): 节后天数，默认0天
    
    返回:
        list: 包含多个日期范围元组的列表 [(start_date, end_date)]
    """
    date_ranges = []
    today = datetime.date.today()
    
    for holiday, years in holiday_dict.items():
        for year, date_str in years.items():
            # 转换字符串为日期对象
            holiday_date = datetime.datetime.strptime(date_str, '%Y-%m-%d').date()
            
            # 计算日期范围
            start_date = holiday_date - datetime.timedelta(days=days_before)
            end_date = holiday_date + datetime.timedelta(days=days_after)
            
            # 确保结束日期不超过昨天
            end_date = min(end_date, today - datetime.timedelta(days=0))
            
            date_ranges.append((start_date, end_date))
    
    return date_ranges

def filter_data_by_dates(df, date_ranges, date_col='date'):
    """
    根据日期范围筛选数据
    
    参数:
        df (pd.DataFrame): 原始数据
        date_ranges (list): 日期范围列表
        date_col (str): 日期列名，默认为'date'
    
    返回:
        pd.DataFrame: 筛选后的数据
    """
    # 转换日期列为日期类型
    df[date_col] = pd.to_datetime(df[date_col]).dt.date
    
    filtered_df = pd.DataFrame()
    
    for start, end in date_ranges:
        mask = (df[date_col] >= start) & (df[date_col] <= end)
        temp_df = df.loc[mask].copy()  # 添加copy()避免SettingWithCopyWarning
        filtered_df = pd.concat([filtered_df, temp_df], ignore_index=True)
    
    # 排序并去重
    filtered_df = filtered_df.sort_values([date_col, 'tic']).reset_index(drop=True)
    filtered_df.drop_duplicates(subset=[date_col, 'tic'], inplace=True)
    
    # 转换回字符串格式
    filtered_df[date_col] = filtered_df[date_col].astype(str)
    
    return filtered_df

# -------------------------------------------------
# 使用示例
# -------------------------------------------------

# 初始化节假日字典（完整版）
holidays = {
    '五一': {
        '2018': '2018-05-01',
        '2019': '2019-05-01',
        '2020': '2020-05-01',
        '2021': '2021-05-01',
        '2022': '2022-05-01',
        '2023': '2023-05-01',
        '2024': '2024-05-01',
        '2025': '2025-05-01'
    },
      '春节': {
        '2018': '2018-02-15',
        '2019': '2019-02-05',
        '2020': '2020-01-24',
        '2021': '2021-02-11',
        '2022': '2022-01-31',
        '2023': '2023-01-21',
        '2024': '2024-02-10',
        '2025': '2025-01-28'
    },   
      '国庆': {
        '2018': '2018-10-01',
        '2019': '2019-10-01',
        '2020': '2020-10-01',
        '2021': '2021-10-01',
        '2022': '2022-10-01',
        '2023': '2023-10-01',
        '2024': '2024-10-01',
        '2025': '2025-10-01'
    }
      
}

# 获取日期范围（示例使用节前30天到节后0天）
date_ranges = get_holiday_date_ranges(
    holiday_dict=holidays,
    days_before=15,
    days_after=10
)

# 打印日期范围
print("生成的日期范围：")
for idx, (start, end) in enumerate(date_ranges, 1):
    print(f"范围{idx}: {start} 至 {end}")

# 示例数据加载（假设已经下载数据）
# portfolio_raw_df = pd.DataFrame({
#     'date': ['2023-04-01', '2023-04-15', '2023-05-01', '2023-05-03'],
#     'tic': ['AAPL', 'AAPL', 'AAPL', 'AAPL'],
#     'close': [170, 175, 172, 178]
# })

# 筛选数据
filtered_df = filter_data_by_dates(
    df=processed,
    date_ranges=date_ranges
)

# 打印结果
print("\n筛选后的数据：")
print(filtered_df)

生成的日期范围：
范围1: 2018-04-16 至 2018-05-11
范围2: 2019-04-16 至 2019-05-11
范围3: 2020-04-16 至 2020-05-11
范围4: 2021-04-16 至 2021-05-11
范围5: 2022-04-16 至 2022-05-11
范围6: 2023-04-16 至 2023-05-11
范围7: 2024-04-16 至 2024-05-11
范围8: 2025-04-16 至 2025-05-02
范围9: 2018-01-31 至 2018-02-25
范围10: 2019-01-21 至 2019-02-15
范围11: 2020-01-09 至 2020-02-03
范围12: 2021-01-27 至 2021-02-21
范围13: 2022-01-16 至 2022-02-10
范围14: 2023-01-06 至 2023-01-31
范围15: 2024-01-26 至 2024-02-20
范围16: 2025-01-13 至 2025-02-07
范围17: 2018-09-16 至 2018-10-11
范围18: 2019-09-16 至 2019-10-11
范围19: 2020-09-16 至 2020-10-11
范围20: 2021-09-16 至 2021-10-11
范围21: 2022-09-16 至 2022-10-11
范围22: 2023-09-16 至 2023-10-11
范围23: 2024-09-16 至 2024-10-11
范围24: 2025-09-16 至 2025-05-02

筛选后的数据：
            date       close        high         low        open    volume  \
0     2021-01-27  251.335556  265.920013  259.950012  265.130005  16050400   
1     2021-01-27  120.430000  125.070000  116.339996  122.169998   7901400   
2     2021-01-27   94.042542  113.000

In [16]:
import datetime
import pandas as pd
# 节假日日期字典（已修正）
holidays = {
    # '春节': {
    #     '2018': '2018-02-15',
    #     '2019': '2019-02-05',
    #     '2020': '2020-01-24',
    #     '2021': '2021-02-11',
    #     '2022': '2022-01-31',
    #     '2023': '2023-01-21',
    #     '2024': '2024-02-10',
    #     '2025': '2025-01-28'
    # },
    '五一': {
        '2018': '2018-05-01',
        '2019': '2019-05-01',
        '2020': '2020-05-01',
        '2021': '2021-05-01',
        '2022': '2022-05-01',
        '2023': '2023-05-01',
        '2024': '2024-05-01',
        '2025': '2025-05-01'
    },
    # '国庆': {
    #     '2018': '2018-10-01',
    #     '2019': '2019-10-01',
    #     '2020': '2020-10-01',
    #     '2021': '2021-10-01',
    #     '2022': '2022-10-01',
    #     '2023': '2023-10-01',
    #     '2024': '2024-10-01',
    #     '2025': '2025-10-01'
    # }
}

# 只选择2019年及以后的日期
selected_holidays = {}
for holiday, years in holidays.items():
    selected_holidays[holiday] = {}
    for year, date_str in years.items():
        if int(year) >= 2019:
            selected_holidays[holiday][year] = date_str

# 生成节假日前后18天的日期范围
date_ranges = []
yesterday = datetime.date.today() - datetime.timedelta(days=0)
for holiday, years in selected_holidays.items():
    for year, date_str in years.items():
        holiday_date = datetime.datetime.strptime(date_str, '%Y-%m-%d').date()
        start_date = holiday_date - datetime.timedelta(days=30)
        end_date = holiday_date + datetime.timedelta(days=0)
        today = datetime.date.today()
        if end_date > yesterday:
            end_date = yesterday
        date_ranges.append((start_date, end_date))


# 找到所有日期范围的最小开始日期和最大结束日期
overall_start_date = min([start for start, end in date_ranges])
overall_end_date = max([end for start, end in date_ranges])

print(f"\n总体下载日期范围：{overall_start_date} 至 {overall_end_date}")



# 下载数据
# logging.info("开始下载股票数据...")
# portfolio_raw_df = YahooDownloader(
#     start_date=str(overall_start_date),
#     end_date=str(overall_end_date),
#     ticker_list=TOP_BRL
# ).fetch_data()
# logging.info("股票数据下载完成。")

# 确保 'date' 列为 datetime 类型
portfolio_raw_df['date'] = pd.to_datetime(portfolio_raw_df['date']).dt.date

# 初始化一个空的DataFrame来存储筛选后的数据
filtered_df = pd.DataFrame()

# 遍历每个日期范围，并筛选数据
for start, end in date_ranges:
    print(start,end)
    mask = (portfolio_raw_df['date'] >= start) & (portfolio_raw_df['date'] <= end)
    temp_df = portfolio_raw_df.loc[mask]
    # print(temp_df.tail())
    filtered_df = pd.concat([filtered_df, temp_df], ignore_index=True)
    print(filtered_df.tail())
filtered_df['date'] = filtered_df['date'].astype(str)
filtered_df = filtered_df.sort_values(['date', 'tic']).reset_index(drop=True)
# 删除重复的数据（如果有重叠的日期范围）
# filtered_df.drop_duplicates(subset=['date', 'tic'], inplace=True)

print("\n筛选后的数据示例：")
print(filtered_df.tail())

print(f"\n筛选后的数据总行数：{len(filtered_df)}")


总体下载日期范围：2019-04-01 至 2025-04-25
2019-04-01 2019-05-01
Empty DataFrame
Columns: [date, close, high, low, open, volume, tic]
Index: []
2020-04-01 2020-05-01
Empty DataFrame
Columns: [date, close, high, low, open, volume, tic]
Index: []
2021-04-01 2021-05-01
Price        date       close        high         low        open     volume  \
163    2021-04-30   70.196838   78.208000   76.370003   76.370003    6386900   
164    2021-04-30   39.840000   41.220001   37.349998   37.730000  116728700   
165    2021-04-30  133.929993  139.580002  133.330002  135.429993    4773800   
166    2021-04-30  268.201630  281.000000  275.200012  276.799988      17355   
167    2021-04-30  349.208893  379.000000  370.000000  377.000000     117655   

Price   tic  
163      JD  
164     NIO  
165     PDD  
166    YANG  
167    YINN  
2022-04-01 2022-05-01
Price        date       close        high         low        open    volume  \
323    2022-04-29   55.950581   65.290001   61.560001   65.080002  19762700 

In [17]:
filtered_df=processed

In [26]:
filtered_df.groupby("tic").count()

,date,close,high,low,open,volume,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,turbulence,vix
tic,,,,,,,,,,,,,,,,
BABA,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248
BILI,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248
FUTU,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248
JD,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248
NIO,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248
PDD,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248
YANG,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248
YINN,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248,248


### Normalize Data

We normalize the data dividing the time series of each stock by its maximum value, so that the dataframe contains values between 0 and 1.

In [28]:
portfolio_norm_df = GroupByScaler(by="tic", scaler=MaxAbsScaler).fit_transform(filtered_df)
portfolio_norm_df

/mnt/data/miniconda3/envs/finstock/lib/python3.10/site-packages/FinRL-0.3.7-py3.10.egg/finrl/meta/preprocessor/preprocessors.py:101: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.21385421 0.13650221 0.19576832 0.20371338 0.4068365  0.38878792
 0.22334883 0.14681225 0.14284705 0.1656536  0.17442074 0.20379998
 0.12464658 0.22262534 0.17670713 0.2049525  0.19610675 0.19211491
 0.14944106 0.16113813 0.19074387 0.15970314 0.11947557 0.10383196
 0.15517434 0.12241616 0.12728072 0.12426419 0.18148375 0.17668181
 0.13342305 0.16318735 0.14056733 0.23006009 0.18503857 0.32455598
 0.33961334 0.48202603 0.25974578 0.26423994 0.3292007  0.4272634
 0.32360332 0.35041371 0.23132053 0.19041877 0.2692897  0.33600922
 0.18696255 0.19820527 0.61337855 0.44634059 0.56218406 0.34805004
 0.19539792 0.38594593 0.35097065 0.37985557 0.29611208 0.36658361
 0.28610049 0.25172078 0.27109509 0.20033976 0.18720238 0.22186322
 0

,date,close,high,low,open,volume,tic,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,turbulence,vix
0,2021-01-27,0.960935,0.969485,0.961354,0.978340,0.213854,BABA,0.527807,0.979791,0.842546,0.862267,0.240338,1.000000,0.953420,0.961779,0.000000,1.000000
1,2021-01-27,0.770161,0.793289,0.784015,0.780888,0.230597,BILI,0.357960,0.871775,0.860246,0.797375,0.030323,0.559192,0.903920,0.923402,0.000000,1.000000
2,2021-01-27,0.503979,0.553244,0.504580,0.504246,0.366613,FUTU,0.287939,0.566386,0.305141,0.872619,0.244771,0.908905,0.524197,0.514855,0.000000,1.000000
3,2021-01-27,0.842908,0.871826,0.851356,0.875844,0.273283,JD,0.138628,0.920053,0.955923,0.645119,-0.012588,0.434908,0.963925,0.968611,0.000000,1.000000
4,2021-01-27,0.909453,0.925542,0.938173,0.926459,0.338426,NIO,0.119941,1.000000,0.924503,0.778005,-0.028256,0.736235,0.990497,0.995927,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1979,2025-05-02,0.350833,0.317758,0.321191,0.322393,0.077691,JD,-0.314173,0.386328,0.387064,0.554836,-0.145853,0.161361,0.421540,0.446925,0.004816,0.661113
1980,2025-05-02,0.064370,0.064241,0.066716,0.065228,0.050890,NIO,0.005615,0.067557,0.054935,0.725015,0.188884,0.050289,0.065223,0.071640,0.004816,0.661113
1981,2025-05-02,0.541465,0.519708,0.538931,0.517533,0.064327,PDD,-0.132988,0.520982,0.536975,0.767124,0.056947,0.112056,0.586335,0.626418,0.004816,0.661113
1982,2025-05-02,0.082507,0.076612,0.078512,0.077828,0.118358,YANG,-0.023742,0.131926,0.102173,0.677861,-0.169290,0.084813,0.116749,0.113852,0.004816,0.661113


In [29]:
df_portfolio = portfolio_norm_df[["date", "tic", "close", "high", "low",'volume','turbulence','macd','vix']]

df_portfolio_train = df_portfolio[(df_portfolio["date"] >= "2019-01-01") & (df_portfolio["date"] <= "2025-04-21")]

df_portfolio_2025 = df_portfolio[(df_portfolio["date"] >= "2023-04-01") & (df_portfolio["date"] <= "2025-05-10")]

unique_dates = df_portfolio_2025['date'].drop_duplicates().sort_values().reset_index(drop=True)
print(unique_dates)

0      2023-04-17
1      2023-04-18
2      2023-04-19
3      2023-04-20
4      2023-04-21
          ...    
119    2025-04-28
120    2025-04-29
121    2025-04-30
122    2025-05-01
123    2025-05-02
Name: date, Length: 124, dtype: object


### Instantiate Environment

Using the `PortfolioOptimizationEnv`, it's easy to instantiate a portfolio optimization environment for reinforcement learning agents. In the example below, we use the dataframe created before to start an environment.

In [30]:
features=["close", "high", "low",'volume','turbulence','vix','macd']
environment = PortfolioOptimizationEnv(
        df_portfolio_train,
        initial_amount=100000,
        comission_fee_pct=0.0025,
        time_window=4,
        features=features,
        normalize_df=None
    )

### Instantiate Model

Now, we can instantiate the model using FinRL API. In this example, we are going to use the EIIE architecture introduced by Jiang et. al.

:exclamation: **Note:** Remember to set the architecture's `time_window` parameter with the same value of the environment's `time_window`.

In [1]:
# set PolicyGradient parameters
model_kwargs = {
    "lr": 0.01,
    "policy": EIIE,
}

# here, we can set EIIE's parameters
policy_kwargs = {
    "k_size": 3,
    "time_window": 4,
    "initial_features":len(features)
}

model = DRLAgent(environment).get_model("pg", device, model_kwargs, policy_kwargs)

model_name = "long_feature"
file_path = f"policy_EIIE_US_{model_name}.pt"
import os
import torch

if market.lower() == "us":
    if os.path.isfile(file_path):
        pass
        model.train_policy.load_state_dict(torch.load(file_path))
        print(f"成功加载模型参数：{file_path}")
    else:
        print(f"未找到模型文件: {file_path}，请确认路径或先进行保存。")
elif market.lower() == "hk":
    if os.path.isfile(file_path):
        model.train_policy.load_state_dict(torch.load("policy_EIIE_HK.pt"))
        print("成功加载模型参数：policy_EIIE_HK.pt")
    else:
        print(f"未找到模型文件: {file_path}，请确认路径或先进行保存。")
elif market.lower() == "jp":
    pass
elif market.lower() == "ch":
    model.train_policy.load_state_dict(torch.load("policy_EIIE_CH3.pt"))
    print("成功加载模型参数：policy_EIIE_CH3.pt")
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

NameError: name 'EIIE' is not defined

### Train Model

In [32]:
DRLAgent.train_model(model, episodes=140)

if market.lower() == "us":  
    torch.save(model.train_policy.state_dict(), "policy_EIIE_US_{}.pt".format(model_name))

elif market.lower() == "hk":
    torch.save(model.train_policy.state_dict(), "policy_EIIE_HK.pt")
elif market.lower() == "ch":
    torch.save(model.train_policy.state_dict(), "policy_EIIE_CH3.pt")
elif market.lower() == "jp":
    torch.save(model.train_policy.state_dict(), "policy_EIIE_JP.pt")
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

  0%|          | 0/140 [00:00<?, ?it/s]

Initial portfolio value:100000
Final portfolio value: 72093.9609375
Final accumulative portfolio value: 0.7209396362304688
Maximum DrawDown: -0.6986531271894252
Sharpe ratio: 0.30718452978371263


  1%|          | 1/140 [00:02<05:21,  2.31s/it]

Initial portfolio value:100000
Final portfolio value: 89292.03125
Final accumulative portfolio value: 0.8929203152656555
Maximum DrawDown: -0.5964401481909422
Sharpe ratio: 0.48939094376750525


  1%|▏         | 2/140 [00:04<05:19,  2.32s/it]

...

 99%|█████████▊| 138/140 [05:20<00:04,  2.28s/it]

Initial portfolio value:100000
Final portfolio value: 180483.953125
Final accumulative portfolio value: 1.8048394918441772
Maximum DrawDown: -0.5964401413865144
Sharpe ratio: 1.193340687284261


 99%|█████████▉| 139/140 [05:22<00:02,  2.27s/it]

Initial portfolio value:100000
Final portfolio value: 180485.03125
Final accumulative portfolio value: 1.8048503398895264
Maximum DrawDown: -0.596440123647314
Sharpe ratio: 1.193348029495473


100%|██████████| 140/140 [05:25<00:00,  2.32s/it]


### Save Model

## Test Model

### Instantiate different environments

Since we have three different periods of time, we need three different environments instantiated to simulate them.

In [33]:
policy = EIIE(time_window=4, device=device,initial_features=len(features))
# policy.load_state_dict(torch.load("policy_EIIE_CH3.pt"))
if market.lower() == "us":
    policy.load_state_dict(torch.load(file_path))
elif market.lower() == "hk":
    policy.load_state_dict(torch.load("policy_EIIE_HK.pt"))
elif market.lower() == "ch":
    policy.load_state_dict(torch.load("policy_EIIE_CH3.pt"))
elif market.lower() == "jp":
    policy.load_state_dict(torch.load("policy_EIIE_JP.pt"))
else:
    raise ValueError("market 变量必须为 'us' 或 'hk'")

environment_2025 = PortfolioOptimizationEnv(
    df_portfolio_2025,
    initial_amount=100000,
    comission_fee_pct=0.0025,
    time_window=4,
    features=features,
    normalize_df=None
)
# df_account_value_ppo, df_actions_ppo = DRLAgent.DRL_prediction(
#     model=model, 
#     environment = environment_2025)
DRLAgent.DRL_validation(model, environment_2025, policy=policy)
# environment_2021 = PortfolioOptimizationEnv(
#     df_portfolio_2021,
#     initial_amount=100000,
#     comission_fee_pct=0.0025,
#     time_window=50,
#     features=["close", "high", "low"],
#     normalize_df=None
# )

# environment_2022 = PortfolioOptimizationEnv(
#     df_portfolio_2022,
#     initial_amount=100000,
#     comission_fee_pct=0.0025,
#     time_window=50,
#     features=["close", "high", "low"],
#     normalize_df=None
# )

# for i, action in enumerate(environment_2025._actions_memory):
#     if not np.isnan(action).all():  # 如果 action 中不全是 NaN
#         if i < len(unique_dates):
#             current_date = unique_dates.iloc[i]
#         else:
#             current_date = '未知日期'  # 处理索引超出范围的情况
#         print(f"Action on {current_date} at step {i}: {action}")

# columns = ["date", "cash"] + TOP_BRL  
# results = []
unique_dates = df_portfolio_2025['date'].drop_duplicates().sort_values().reset_index(drop=True)

columns = ["date", "cash"] + TOP_BRL  
results = []

shift_value = 3  # 想统一日期往后移 4 格
actions = environment_2025._actions_memory

for i, action in enumerate(actions):
    # 如果 action 全是 NaN，我们跳过
    if np.isnan(action).all():
        continue
    
    shifted_index = i + shift_value
    if shifted_index < len(unique_dates):
        current_date = unique_dates.iloc[shifted_index]
    else:
        current_date = '未知日期'
    
    # 将 action 转成百分比字符串，比如 0.123 -> "12.30%"
    action_in_percent = [f"{x*100:.2f}%" for x in action]
    
    # 构建一行 [日期, 第一列现金比例, 后面的列是各股票比例]
    row = [current_date] + action_in_percent
    results.append(row)

# 创建 DataFrame
df_action_percent = pd.DataFrame(results, columns=columns)
print("日期统一往后移 3 格后的 DataFrame：")
print(df_action_percent.tail(10))



Initial portfolio value:100000
Final portfolio value: 133842.59375
Final accumulative portfolio value: 1.3384259939193726
Maximum DrawDown: -0.5964379435279865
Sharpe ratio: 1.1790334096377295
日期统一往后移 3 格后的 DataFrame：
           date     cash   BILI    NIO     JD   YINN   YANG   FUTU   BABA  \
111  2025-04-21  100.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%   
112  2025-04-22  100.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%   
113  2025-04-23  100.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%   
114  2025-04-24  100.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%   
115  2025-04-25  100.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%   
116  2025-04-28  100.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%   
117  2025-04-29  100.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%   
118  2025-04-30  100.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%   
119  2025-05-01  100.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%  0.00%   
120  2025-05

In [ ]:
# unique_dates = df_portfolio['date'].drop_duplicates().sort_values().reset_index(drop=True)


### Test EIIE architecture
Now, we can test the EIIE architecture in the three different test periods. It's important no note that, in this code, we load the saved policy even though it's not necessary just to show how to save and load your model.

In [21]:
EIIE_results = {
    "training": environment._asset_memory["final"],
    "2025": {},

}

# instantiate an architecture with the same arguments used in training
# and load with load_state_dict.
policy = EIIE(time_window=50, device=device)
policy.load_state_dict(torch.load("policy_EIIE_US.pt"))

# 2020
DRLAgent.DRL_validation(model, environment_2025, policy=policy)
EIIE_results["2025"]["value"] = environment_2025._asset_memory["final"]

# # 2021
# DRLAgent.DRL_validation(model, environment_2021, policy=policy)
# EIIE_results["2021"]["value"] = environment_2021._asset_memory["final"]

# # 2022
# DRLAgent.DRL_validation(model, environment_2022, policy=policy)
# EIIE_results["2022"]["value"] = environment_2022._asset_memory["final"]

RuntimeError: Error(s) in loading state_dict for EIIE:
	size mismatch for sequential.0.weight: copying a param with shape torch.Size([2, 4, 1, 3]) from checkpoint, the shape in current model is torch.Size([2, 3, 1, 3]).
	size mismatch for sequential.2.weight: copying a param with shape torch.Size([20, 2, 1, 2]) from checkpoint, the shape in current model is torch.Size([20, 2, 1, 48]).

In [ ]:
# unique_dates = df_portfolio['date'].drop_duplicates().sort_values().reset_index(drop=True)
unique_dates = df_portfolio_2025['date'].drop_duplicates().sort_values().reset_index(drop=True)
for i, action in enumerate(environment_2025._actions_memory):
    if not np.isnan(action).all():  # 如果 action 中不全是 NaN
        if i < len(unique_dates):
            current_date = unique_dates.iloc[i]
        else:
            current_date = '未知日期'  # 处理索引超出范围的情况
        print(f"Action on {current_date} at step {i}: {action}")

### Test Uniform Buy and Hold
For comparison, we will also test the performance of a uniform buy and hold strategy. In this strategy, the portfolio has no remaining cash and the same percentage of money is allocated in each asset.

In [ ]:
filtered_df.tail()

In [ ]:
UBAH_results = {
    "train": {},
    "2025": {},
    # "2021": {},
    # "2022": {}
}

PORTFOLIO_SIZE = len(TOP_BRL)

# train period
terminated = False
environment.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = environment.step(action)
UBAH_results["train"]["value"] = environment._asset_memory["final"]

# 2020
terminated = False
environment_2025.reset()
while not terminated:
    action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
    _, _, terminated, _ = environment_2025.step(action)
UBAH_results["2025"]["value"] = environment_2025._asset_memory["final"]

# # 2021
# terminated = False
# environment_2021.reset()
# while not terminated:
#     action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
#     _, _, terminated, _ = environment_2021.step(action)
# UBAH_results["2021"]["value"] = environment_2021._asset_memory["final"]

# # 2022
# terminated = False
# environment_2022.reset()
# while not terminated:
#     action = [0] + [1/PORTFOLIO_SIZE] * PORTFOLIO_SIZE
#     _, _, terminated, _ = environment_2022.step(action)
# UBAH_results["2022"]["value"] = environment_2022._asset_memory["final"]

### Plot graphics

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline 

plt.plot(UBAH_results["train"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["training"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in training period")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2025"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2025"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2025")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2021"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2021"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2021")
plt.legend()

plt.show()

In [ ]:
plt.plot(UBAH_results["2022"]["value"], label="Buy and Hold")
plt.plot(EIIE_results["2022"]["value"], label="EIIE")

plt.xlabel("Days")
plt.ylabel("Portfolio Value")
plt.title("Performance in 2022")
plt.legend()

plt.show()

We can see that the agent is able to learn a good policy but its performance is worse the more the test period advances into the future. To get a better performance in 2022, for example, the agent should probably be trained again using more recent data.